-character level language model
-trained data of characters
-at test time  keeps predicting next charcter

-ex:
-  suresh
-   ramesh
-   sunil
-   suraj
-   naresh
-   here might predict .sh.

-let the menatl model for now be each name is sentence each charater is a word

-in n gram models
-given a sentence p(w1, w2, w3, w4) = p(w1)p(w2|w1)p(w3|w1,w2)p(w4|w1,w2,w3)
-similarly for a given 


For a batch size of $N$ and $C$ classes, the multi-class cross-entropy loss is $\mathcal{L}_{\text{batch}} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{c=1}^{C} y_{i,c} \log(p_{i,c})$.

https://share.gemini.google/1g2DMTImQ4bE

An **$n$-gram model** is a classical statistical language model that predicts the next word by looking back at only the preceding $n-1$ words, estimating probabilities purely through frequency counts.

It directly addresses the probability chain rule discussed earlier, simplifying it using the **Markov assumption**.

---

### 1. From the Chain Rule to the Markov Assumption

Recall the exact chain rule of language modeling:

$$P(w_1, w_2, \dots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_1, w_2, \dots, w_{t-1})$$

In a 50-word sentence, calculating the 50th word requires conditioning on a 49-word ordered tuple:

$$P(w_{50} \mid w_1, w_2, \dots, w_{49})$$

In traditional statistics, you cannot estimate this probability. That specific 49-word sequence has almost certainly never appeared before in human writing, meaning its count in your training data is 0.

#### The $n$-Gram Approximation

An $n$-gram model discards distant history, assuming that the probability of the current word depends **only on the most recent $n-1$ words**:

$$P(w_t \mid w_1, w_2, \dots, w_{t-1}) \approx P(w_t \mid w_{t-n+1}, \dots, w_{t-1})$$

| Model | $n$ | Looks Back | Mathematical Form |
| --- | --- | --- | --- |
| **Unigram** | 1 | 0 words (bag of words) | $P(w_t)$ |
| **Bigram** | 2 | 1 word | $P(w_t \mid w_{t-1})$ |
| **Trigram** | 3 | 2 words | $P(w_t \mid w_{t-2}, w_{t-1})$ |
| **4-gram** | 4 | 3 words | $P(w_t \mid w_{t-3}, w_{t-2}, w_{t-1})$ |

Notice that the condition $(w_{t-n+1}, \dots, w_{t-1})$ remains an **ordered tuple**. A bigram model treats $(w_1 = \text{"dog"}, w_2 = \text{"barks"})$ as completely distinct from $(w_1 = \text{"barks"}, w_2 = \text{"dog"})$.

---

### 2. How an $n$-Gram Model "Trains" (Counting MLE)

Unlike neural networks, which use gradient descent and backpropagation, an $n$-gram model trains via direct **frequency counting** using Maximum Likelihood Estimation (MLE):

$$P_{\text{MLE}}(w_t \mid w_{t-n+1}, \dots, w_{t-1}) = \frac{\text{Count}(w_{t-n+1}, \dots, w_{t-1}, w_t)}{\text{Count}(w_{t-n+1}, \dots, w_{t-1})}$$

#### Concrete Example (Bigram Model)

Suppose our entire training corpus consists of three short sentences:

1. `<s> the cat sat on the mat </s>`
2. `<s> the dog sat on the rug </s>`
3. `<s> the cat ate the food </s>`

*(where `<s>` and `</s>` mark the start and end of a sentence)*

To calculate $P(\text{"sat"} \mid \text{"cat"})$:

1. Count how many times the prefix `"cat"` appears: $\text{Count}(\text{"cat"}) = 2$.
2. Count how many times the bigram `("cat", "sat")` appears: $\text{Count}(\text{"cat", "sat"}) = 1$.
3. Divide:

$$P(\text{"sat"} \mid \text{"cat"}) = \frac{\text{Count}(\text{"cat", "sat"})}{\text{Count}(\text{"cat"})} = \frac{1}{2} = 0.5$$



To calculate $P(\text{"ate"} \mid \text{"cat"})$:


$$P(\text{"ate"} \mid \text{"cat"}) = \frac{\text{Count}(\text{"cat", "ate"})}{\text{Count}(\text{"cat"})} = \frac{1}{2} = 0.5$$

---

### 3. The Three Structural Breakdowns of $n$-Grams

While computationally trivial to train, $n$-gram models hit three severe mathematical ceilings that neural networks eventually solved:

#### A. The Curse of Dimensionality

If your vocabulary has $\vert{}V\vert{} = 50,000$ words:

* A bigram model must track a table of $\vert{}V\vert{}^2 = 2.5 \times 10^9$ possible pairs.
* A trigram model requires $\vert{}V\vert{}^3 = 1.25 \times 10^{14}$ possible triplets.
* A 5-gram model requires $\vert{}V\vert{}^5 \approx 3.1 \times 10^{23}$ values (more than all storage on Earth).

Because memory requirements scale exponentially as $O(\vert{}V\vert{}^n)$, $n$-gram models rarely scaled beyond $n = 5$.

#### B. The Zero-Probability / Sparsity Problem

If the test sentence is `"the dog ate the bone"`, but the training corpus never saw the bigram `("dog", "ate")`:

$$\text{Count}(\text{"dog", "ate"}) = 0 \implies P(\text{"ate"} \mid \text{"dog"}) = 0$$

Because the total sentence probability is a product of probabilities:

$$P(\text{Sentence}) = P(\text{"dog"} \mid \text{"the"}) \times 0 \times \dots = 0$$

A single novel word pair causes the entire sentence probability to crash to zero. Researchers had to invent complex heuristic patches called **smoothing** (Laplace, Good-Turing, and Kneser-Ney smoothing) to steal small fractions of probability mass from frequent words and assign them to unseen combinations.

#### C. Zero Semantic Generalization

In an $n$-gram model, words are discrete, orthogonal symbols (atomic indices):

```text
"cat" = Token #1042
"feline" = Token #8911
"dog" = Token #2051

```

If the model has seen `"the cat drank the milk"` 10,000 times, but has never seen `"the feline drank the milk"`, it assigns $0$ probability to the second sentence. It has no mathematical mechanism to recognize that a *cat* and a *feline* share behavioral or grammatical properties.

---

### How Transformers Compare to $n$-Grams

| Property | $n$-Gram Models | Modern Transformers |
| --- | --- | --- |
| **History Window ($n$)** | Fixed and small ($n \le 5$) | Massive ($2,048$ to $1,000,000+$ tokens) |
| **Parameter Scaling** | Exponential: $O(\Vert{}V\Vert{}^n)$ | Linear/Quadratic with dimension & context |
| **Word Representation** | Discrete strings / IDs | Continuous vectors (embeddings) |
| **Generalization** | Exact-match counting only | Semantic similarity via dot products |
| **Unseen Phrasing** | Assigns zero (needs smoothing) | Generalizes naturally via vector proximity |

In [119]:
words = open("./names.txt", "r").read().splitlines()

In [120]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [121]:
print(min(len(w) for w in words))
print(max(len(w) for w in words))

2
15


In [122]:
for word in words[:1]:
    for i in range(len(word)):
        if i < (len(word)-1):
            print(word[i],word[i+1])

e m
m m
m a


In [123]:
for word in words[:1]:
    for i,j in zip(word,word[1:]):
        print(i,j)

e m
m m
m a


ther is more info we are missing<br>
not accounting for the starting and ending letter likelihood<br>
so

In [124]:
for word in words[:1]:
    chars = ["<S>"] + list(word)  + ["<E>"]
    for ch1, ch2 in zip(chars, chars[1:]):
        print(ch1, ch2)

<S> e
e m
m m
m a
a <E>


In [125]:
chars = sorted(list(set("".join(words))))
print(chars)
print(len(chars))

['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
26


In [126]:
ctoi = {}
for i,c in enumerate(chars, start=1):
    ctoi[c] = i
ctoi["."] = 0
ctoi

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '.': 0}

In [127]:
import torch

In [128]:
N = torch.zeros((27, 27), dtype = torch.int32)

In [129]:
for word in words:
    chars = ["."] + list(word) + ["."]
    for ch1, ch2 in zip(chars, chars[1:]):
        N[ctoi[ch1], ctoi[ch2]] += 1

In [130]:
N

tensor([[   0, 4410, 1306, 1542, 1690, 1531,  417,  669,  874,  591, 2422, 2963,
         1572, 2538, 1146,  394,  515,   92, 1639, 2055, 1308,   78,  376,  307,
          134,  535,  929],
        [6640,  556,  541,  470, 1042,  692,  134,  168, 2332, 1650,  175,  568,
         2528, 1634, 5438,   63,   82,   60, 3264, 1118,  687,  381,  834,  161,
          182, 2050,  435],
        [ 114,  321,   38,    1,   65,  655,    0,    0,   41,  217,    1,    0,
          103,    0,    4,  105,    0,    0,  842,    8,    2,   45,    0,    0,
            0,   83,    0],
        [  97,  815,    0,   42,    1,  551,    0,    2,  664,  271,    3,  316,
          116,    0,    0,  380,    1,   11,   76,    5,   35,   35,    0,    0,
            3,  104,    4],
        [ 516, 1303,    1,    3,  149, 1283,    5,   25,  118,  674,    9,    3,
           60,   30,   31,  378,    0,    1,  424,   29,    4,   92,   17,   23,
            0,  317,    1],
        [3983,  679,  121,  153,  384, 1271,   82,

RuntimeError: a Tensor with 729 elements cannot be converted to Scalar